<a href="https://colab.research.google.com/github/Tamanna-op/flyrank-ml-internship-task/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamanna-op/flyrank-ml-internship-task/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
# Reconnect DuckDB
import duckdb

con = duckdb.connect()

# Use your Colab Secret named HF_TOKEN
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE SECRET hf_secret ("
    f"TYPE HUGGINGFACE, "
    f"TOKEN '{HF_TOKEN}'"
    f")"
)

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_path = f"{rel}/fact_content_daily_performance.parquet"
content_path = f"{rel}/dim_content.parquet"

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [26]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

rel = "hf://datasets/FlyRank/internship-warehouse"

fact_path = f"{rel}/fact_content_daily_performance.parquet"
content_path = f"{rel}/dim_content.parquet"

MONTH = "2026-03"

## 1. Unit of analysis + time window

Unit of analysis: one content item for one client during the selected monthly observation window.

For this notebook, I will use March 2026 as the development window.

The main fact table is fact_content_daily_performance, which records daily performance for a client-content pair. I will aggregate those daily observations to the client-content level for March.

I will join dim_content when content metadata is needed.

The decision supported by this data is: which pages should be reviewed first for refresh, expansion, protection, pruning/consolidation review, or monitoring?

I will use only information that would be available at the decision moment. Future outcome information will not be used as a feature.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### Features

I will use five observable features:

- gsc_impressions — search visibility/exposure during the March observation window.
- gsc_avg_position — observed average search position during the March observation window.
- ctr — calculated from observed clicks and impressions during March.
- content_age_days — calculated from the content creation date and the end of the March window.
- days_since_last_update — calculated from the content update date and the end of the March window.

### Label / proxy

For this early warehouse experiment, I will treat a future decline outcome as the eventual target direction I want to investigate. I will not create a future label in this notebook yet because this assignment is focused on establishing the data contract and feature availability.

### Context

client_hash_id and content_hash_id are used to identify and join records. They are not predictive features because the hash values themselves have no meaningful numerical interpretation.

### Excluded

I will deliberately exclude identifiers such as client_hash_id, content_hash_id, url_hash_id, and keyword_hash_id from the feature set. I will also exclude any product decision/output such as a priority score or action flag because such information could duplicate the decision the model is supposed to support or introduce leakage.

I will also avoid using future-window measurements as features.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Verify the grain

The fact table is documented as daily performance at the report-date, client, and content level. I will verify that (report_date, client_hash_id, content_hash_id) behaves as the row grain by checking for duplicate combinations.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS distinct_grain_rows
FROM read_parquet('{fact_path}')
WHERE month = '{MONTH}'
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


### Query 2 — Verify March row count and date span

I will verify how many fact-table observations exist for March 2026 and the actual report-date range present in that slice.

In [39]:
count_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{fact_path}')
WHERE month = '{MONTH}'
""").df()

display(count_window)

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Verify GSC availability

The refresh lane relies heavily on search-performance signals, so I will check how many March rows have GSC data available.

I use IS TRUE explicitly so that only rows where availability is positively confirmed are counted as available.

In [40]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_march_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_with_gsc_available,

    COUNT(*) FILTER (
        WHERE gsc_impressions IS NOT NULL
    ) AS rows_with_impressions

FROM read_parquet('{fact_path}')
WHERE month = '{MONTH}'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,rows_with_gsc_available,rows_with_impressions
0,9841378,3611061,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.